In [ ]:
!pip install qdrant-client sentence-transformers

In [ ]:
from qdrant_client import QdrantClient, models
from sentence_transformers import SentenceTransformer

/usr/local/lib/python3.10/dist-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


In [ ]:
# Qdrant 클라이언트 초기화
QDRANT_URL = "https://6e46b2c2-f28a-4f28-854d-432ab699fdfd.europe-west3-0.gcp.cloud.qdrant.io"
QDRANT_API_KEY = "u2eejPgTwIyhr7BVjFBtkjGdGYPWvzQTBkoYycErtm5cyrFjwEEH9w"

client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)

In [ ]:
# 임베딩 모델 초기화
encoder = SentenceTransformer("jhgan/ko-sroberta-multitask")

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
def search_disease_info(query_text, limit=3):
    # 쿼리 텍스트를 벡터로 변환
    query_vector = encoder.encode(query_text).tolist()

    # Qdrant에서 검색 수행
    search_results = client.search(
        collection_name='son5',  # 컬렉션 이름이 'son5'인지 확인해주세요
        query_vector=query_vector,
        limit=limit
    )

    # 결과 처리
    results = []
    for hit in search_results:
        result = {
            "질병": hit.payload.get("질병", "N/A"),
            "답변": hit.payload.get("답변", "N/A"),
            "유사도 점수": hit.score
        }
        results.append(result)

    return results

In [ ]:
# 테스트 함수
def test_search():
    while True:
        query = input("질문을 입력하세요 (종료하려면 '끝내기' 입력): ")
        if query.lower() == '끝내기':
            print("검색을 종료합니다.")
            break

        results = search_disease_info(query)
        print(f"\n'{query}'에 대한 검색 결과:\n")
        for i, result in enumerate(results, 1):
            print(f"결과 {i}:")
            print(f"질병: {result['질병']}")
            print(f"답변: {result['답변']}")
            print(f"유사도 점수: {result['유사도 점수']:.4f}")
            print("-" * 50)

# 테스트 실행
test_search()

질문을 입력하세요 (종료하려면 '끝내기' 입력): 운동 전후 종아리 뒤쪽 통증이 있고 발목을 움직일때 소리가 나

'운동 전후 종아리 뒤쪽 통증이 있고 발목을 움직일때 소리가 나'에 대한 검색 결과:

결과 1:
질병: 아킬레스 건염
답변: 아킬레스 건염은 주로 발목 부근에서 발생하는 염증성 질환으로 알려져 있습니다. 아킬레스 건염은 발목 주변에서 통증과 부종을 동반하는 통증이 주로 나타납니다. 운동과 같은 활동 후 발뒤꿈치의 아킬레스 건 부분에서 염증이 발생하면 발목 주변에 강한 통증이 생길 수 있습니다. 또한, 아킬레스 건염은 종아리로 통증이 전해질 수 있어 발목 부분을 걷거나 달리는 동안 통증이 느껴질 수 있습니다. 만약 아킬레스 건염을 의심한다면, 휴식을 취해도 통증이 지속되는 경우 의사의 진료를 받는 것이 중요합니다. 아킬레스 건염은 다양한 증상을 동반하는 염증성 질환입니다. 만약 발목 주변에서 통증이 나타나면 의사의 진료를 받아야 합니다. 조기에 대처하지 않으면 심각한 합병증을 초래할 수 있으므로, 가능한 빠른 시일 내에 증상을 인지하고 의사의 도움을 받는 것이 중요합니다.
유사도 점수: 0.8145
--------------------------------------------------
결과 2:
질병: 아킬레스 건염
답변: 아킬레스 건염은 아킬레스 건 부위에 염증이 생기는 질환으로, 주로 스포츠 활동이나 잘못된 운동 방법으로 인해 발생합니다. 이로 인해 통증이 나타나며, 아킬레스 건염의 주요 증상은 다음과 같습니다. 1. 통증 초기에는 활동 중 통증이 심해지고, 일정 수준의 움직임에 의해서도 계속되는 경우가 있습니다.2. 부종 통증과 함께 발뒤꿈치 부위에 부종이 발생하여 신발 신기가 힘들어질 수 있습니다.3. 염증 아킬레스 건 부위에 염증이 생기면 운동 시 통증이 더욱 심해지고, 종아리 부분까지 염증이 퍼질 수 있습니다.4. 보행 문제 운동을 하지 않을 때에도 통증이 지속되는 경우도 있습니다. 아킬레스 건염은 적절한 예방과 치료가 필요한 상황입

Qdrant 확인 결과 :
 - 질문이 명확하다면 답변 역시 명확하게 대부분 잘 나오는 것으로 보임
 - 다만 질문이 애매하다면(3번째 질문) 답변 역시 많이 갈리는 것으로 보임(애매해 보임)


그래도 정말 빠르게 답변을 가져오는 것을 확인할 수 있었음.